# List the tables in the database to verify

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行
# set current working directory to the location of the database file
os.chdir('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/code')

# ---------------------------------------
# 宏观因子 -- Download from Michael W. McCracken 
# ---------------------------------------

In [ ]:
target_timeline = pd.date_range(start='1991-01-01', end='2024-12-31', freq='ME')
macro_config = pd.read_excel('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor.xlsx', sheet_name='McCracken', index_col=0)
macro_cols = macro_config.index.dropna().unique().tolist()
# 构建承载最终结果的大面板 (初始化全为 NaN)
master_panel = pd.DataFrame(index=target_timeline, columns=macro_cols)
master_panel.index.name = 'mthcaldt'

print(f">>> Target Timeline: 1991-01 to 2024-12 ({len(target_timeline)} months)")
vintages_dir='/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/Historical FRED-MD Vintages Final'
output_save_path='/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data'

print(f">>> 开始动态构建宏观大表 (共 {len(target_timeline)} 个月)...")

for cutoff_date in target_timeline:
    # A. 1999年8月前借用 1999-08.csv，之后直接对齐 YYYY-MM.csv
    if cutoff_date < pd.to_datetime('1999-08-31'):
        continue
    else:
        filename = cutoff_date.strftime('%Y-%m') + ".csv"
        
    file_path = os.path.join(vintages_dir, filename)

    # B. 读入单张快照大表
    df_raw_v = pd.read_csv(file_path)
    transform_dict = df_raw_v.iloc[0].to_dict() # 转换指令
    df_vintage = df_raw_v.iloc[1:].copy()        # 真实数据
    
    df_vintage['sasdate'] = pd.to_datetime(df_vintage['sasdate']) + pd.offsets.MonthEnd(0)  # 确保日期是月末
    df_vintage.set_index('sasdate', inplace=True)
    
    # 强制转成浮点数
    df_vintage = df_vintage.apply(pd.to_numeric, errors='coerce')
    
    # C. 自动化平稳化变形 (根据指令自动洗干净整张表的历史)
    for col in macro_cols:
        if col in df_vintage.columns:
            try: t_code = int(float(transform_dict[col]))
            except: t_code = 1
            
            if t_code == 1 or col in ['GS10', 'TB3MS', 'BAA']: pass
            elif t_code == 2: df_vintage[col] = df_vintage[col].diff()
            elif t_code == 3: df_vintage[col] = df_vintage[col].diff().diff()
            elif t_code == 4: df_vintage[col] = np.log(df_vintage[col])
            elif t_code == 5: df_vintage[col] = np.log(df_vintage[col]).diff()
            elif t_code == 6: df_vintage[col] = np.log(df_vintage[col]).diff().diff()
            elif t_code == 7:
                df_vintage[col] = (
                    df_vintage[col]
                    .pct_change(fill_method=None)
                    .diff()
                )
            else:
                raise ValueError(f"Unknown transformation code: {t_code}")

            current_view = master_panel.loc[:cutoff_date, col]
            n_missing = current_view.isna().sum() 

            valid_vintage_data = df_vintage[col].dropna()
            n_take = min(n_missing, len(valid_vintage_data))
            # copy the last n_missing valid data points from the vintage to fill the missing values in the master panel without using index
            master_panel.loc[current_view[current_view.isna()].index, col] = valid_vintage_data.tail(n_take).values

# ==========================================================
# 🛠️ 衍生计算：在终极无偏大表生成后，横向合成两个利差
# ==========================================================
print(">>> 正在横向合成期限利差与信用利差...")
MacroFactor_McCracken = master_panel.reset_index()

MacroFactor_McCracken['TERM_SPREAD'] = MacroFactor_McCracken['GS10'] - MacroFactor_McCracken['TB3MS']
MacroFactor_McCracken['CREDIT_SPREAD'] = MacroFactor_McCracken['BAA'] - MacroFactor_McCracken['GS10']

# 计算完利差后，物理扔掉绝对值指标，保持特征纯净
MacroFactor_McCracken.drop(columns=['GS10', 'TB3MS', 'BAA'], inplace=True, errors='ignore')

# 导出为parquet
MacroFactor_McCracken.to_parquet(os.path.join(output_save_path, 'MacroFactor_McCracken.parquet'), index=False)

print(f"Total missing values: {MacroFactor_McCracken.isna().sum().sum()}")
display(pd.concat([MacroFactor_McCracken.head(4), MacroFactor_McCracken.tail(4)]))

>>> Target Timeline: 1991-01 to 2024-12 (408 months)
>>> 开始动态构建宏观大表 (共 408 个月)...
>>> 正在横向合成期限利差与信用利差...
Total missing values: 0


,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1991-01-31,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
1,1991-02-28,-0.001892,0.1,-0.041536,-0.004931,-0.003955,-0.029746,0.02,6.682109,0.001999,-0.093898,-0.007471,-0.4,-0.7,-0.3,-0.000014,26.9318,3.54,1.87,2.36
2,1991-03-31,-0.002384,0.2,0.091651,-0.007933,-0.009491,0.00533,0.02,6.872128,0.000704,0.100378,0.005295,-0.66,-0.61,0.5,-0.002975,22.1126,3.82,1.91,2.22
3,1991-04-30,-0.001568,0.2,0.024146,-0.009247,0.006162,-0.061953,0.0,6.82546,-0.000088,0.043726,-0.002343,-0.13,-0.08,0.2,-0.000742,19.041,3.97,2.2,1.98
4,1991-05-31,-0.001709,-0.1,-0.082863,0.003497,0.011971,0.041812,0.0,6.908755,-0.002301,0.034535,-0.002986,-0.21,-0.34,0.6,0.002221,18.9272,4.03,2.39,1.9


# --------------------------------------------
# 微观因子 -- WRDS CRSP Ratios 
# --------------------------------------------

这是量化基本面研究中最正统、也最纯粹的武器库——**WRDS Financial Ratios Suite (WRDS 官方财务比率库)**。

对于你目前暂时搁置 OAP 异象数据、决定专心用基本面和宏观数据跑通 MDN 基线模型（Baseline）的战略来说，这张表简直完美。它最大的价值在于：WRDS 官方已经帮你把极其复杂的底层财报科目，计算成了**对冲基金最常用的标准化比率**，并且完美对齐了 `public_date`（无前视偏差）。

面对这近 90 个变量，你绝对不能全塞进神经网络（会面临严重的多重共线性和过拟合）。结合你预测 **$ES_{5\%}$ (极左尾崩盘风险)** 的目标，我为你将这些指标分为 **5 大核心战区**，并为你挑出 MDN 最需要的“尾部特征”。

---

### 🚨 战区一：债务与破产雷区 (Solvency & Leverage) - **最核心的左尾触发器**
对于预测极值崩盘，**“公司欠了多少钱，还不还得上”**永远是第一位的。高杠杆公司在宏观环境恶化时，最容易出现断崖式下跌。

* **`short_debt` (短期债务 / 总债务):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 极度致命。短债比例越高，资金链断裂的风险呈指数级上升。
* **`intcov_ratio` (利息保障倍数):** **【MDN 必选 ⭐⭐⭐⭐】** EBIT / 利息支出。衡量公司赚的钱够不够还利息。低于 1 的公司随时暴雷。
* **`debt_at` (总债务 / 总资产):** **【MDN 必选 ⭐⭐⭐】** 最经典的宏观杠杆率。
* **`debt_ebitda` (总债务 / EBITDA):** 华尔街信用评级最看重的指标，比 `debt_at` 更能反映真实的偿债压力。
* *其他同类冗余指标 (建议剔除其一):* `lt_debt`, `debt_capital`, `de_ratio`, `totdebt_invcap`（选 `debt_at` 即可，没必要全放）。

### 🩸 战区二：流动性与现金流 (Liquidity & Cash Flow) - **抗风险的护城河**
当市场恐慌时，利润只是会计数字，**现金才是命**。缺乏流动性的公司在遭遇做空或行业寒冬时毫无抵抗力。

* **`ocf_lct` (经营现金流 / 流动负债):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 这是最真实的短期生存指标。赚不到真金白银还要面临短期债务的公司，左尾极厚。
* **`cash_ratio` (现金比率):** **【MDN 必选 ⭐⭐⭐⭐】** 账面纯现金 / 流动负债。直接衡量“今天如果债主逼债，能不能活下来”。
* **`fcf_ocf` (自由现金流 / 经营现金流):** 反映公司的资本开支压力。
* *其他同类冗余指标:* `curr_ratio` (流动比率), `quick_ratio` (速动比率)。在预测崩盘时，它们不如 `cash_ratio` 狠，因为存货和应收账款在破产时往往一文不值。

### 🎭 战区三：盈余质量 (Accounting Quality) - **财务造假与爆雷预测**
* **`accrual` (应计项目 / 平均资产):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 盈余操纵的试金石！如果一家公司净利润很高，但全在应收账款和存货里（高 accrual），这是未来业绩大爆雷（股价单日腰斩）的最强预兆。

### 💰 战区四：估值与均值回归 (Valuation & Multiples) - **决定下跌的空间**
估值越高，泡沫破裂时的“重力加速度”越大。估值因子是决定收益率分布中心 $\mu$ 和方差 $\sigma$ 的重要锚点。

* **`bm` (账面市值比):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 最经典的价值因子。
* **`pe_exi` (扣非市盈率):** **【MDN 必选 ⭐⭐⭐⭐】** 剔除了非经常性损益的真实市盈率，比 `pe_inc` 更干净。
* **`evm` (企业价值倍数 EV/EBITDA):** 并购和私募股权最爱看的指标，不受资本结构（负债率）影响。
* *其他同类冗余指标:* `pe_op_basic`, `ptb`, `ps` (市销率), `pcf` (市现率)。选 2-3 个核心的即可。

### ⚙️ 战区五：盈利能力与效率 (Profitability & Efficiency)
* **`roe` (净资产收益率):** **【MDN 必选 ⭐⭐⭐⭐】** 基本面质量的终极体现。
* **`gpm` (毛利率) 或 `opmad` (税后营业利润率):** 衡量公司在行业内的护城河深度。毛利被压缩通常是崩盘的早期信号。
* **`at_turn` (总资产周转率):** 运营效率。
* *其他同类冗余指标:* `roa`, `roce`, `aftret_eq` 等高度相关，留 `roe` 足矣。

---

### 🗑️ 垃圾桶：绝对不要放入神经网络的变量

1.  **标识符与日期：** `gvkey`, `adate`, `qdate`, `public_date`（仅用于合并和排序，绝对不能作为 $X$ 输入网络）。
2.  **文本与描述：** `gicdesc`, `ffi48_desc` 等一切 `VARCHAR` 文本列。
3.  **绝对数值：** `mktcap` (总市值), `price` (股价)。**如果你要放，必须对它们进行横截面 Rank 转换**，否则几万亿的市值会瞬间引起梯度爆炸。

---

### 🚀 你的 MDN 基线特征包 (Baseline Feature Set) 建议

为了避免维度灾难，我建议你从这张表里**精挑细选这 15 个黄金比率**作为 $X_{Fundamentals}$：

```python
mdn_features_wrds = [
    # 破产与债务
    'short_debt', 'intcov_ratio', 'debt_at', 'debt_ebitda',
    # 流动性与造假预警
    'ocf_lct', 'cash_ratio', 'accrual', 
    # 估值泡沫
    'bm', 'pe_exi', 'evm', 'divyield',
    # 盈利与效率
    'roe', 'gpm', 'at_turn', 'rd_sale'  # rd_sale (研发占比) 对美股科技股防雷很有用
]
```

将这 15 个特征，加上你处理好的 Goyal 宏观特征（如 `TAIL`, `DEF`, `VIXCLS`），你的第一版输入层 $X_T$ 就能控制在 20 个维度左右。配合上横截面 Rank 标准化，这是一个极其清爽且具备极强经济学解释力的输入矩阵！

这些财务比率中必定存在大量的 `NaN`（比如很多科技公司没有短期债务，或者尚未盈利导致市盈率缺失）。面对这些残缺，在进行 Rank 转换之前，你打算使用全市场的截面中位数填充，还是基于它所在的行业 (`gsector` / `ffi48`) 进行同行业中位数填充？

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import os
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行
wrds_path = '/Users/jianbinchen/NonSync/GitHub/Research/DataSet/WRDS.sqlite'
# set current working directory to the location of the database file
os.chdir('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/code')

In [4]:
crsp_wrds_msfv2_query_1990=pd.read_parquet('../data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow')
pd.concat([crsp_wrds_msfv2_query_1990.head(3), crsp_wrds_msfv2_query_1990.tail(3)], axis=0)

,permno,mthcaldt,mthret,mthretx,mthprc,mthcap,mthvol,shrout,naics,siccd,mthdelflg,delreasontype,year
0,10001,1990-01-31,-0.018519,-0.018519,9.9375,1.015613e+04,3.525500e+04,1022,0,4920,N,UNAV,1990
1,72996,1990-01-31,-0.106383,-0.106383,5.2500,1.897875e+04,7.820000e+04,3615,0,6211,N,NACT,1990
2,72988,1990-01-31,-0.076923,-0.076923,1.5000,2.658000e+03,1.315200e+04,1772,0,2099,N,LP,1990
2198471,21118,2025-12-31,-0.040892,-0.040892,2.5800,2.884208e+05,4.764334e+07,111791,424350,5137,N,NACT,2025
2198472,84397,2025-12-31,0.036761,0.033780,51.7200,5.192171e+05,9.384600e+05,10039,522110,6022,N,NACT,2025
2198473,93436,2025-12-31,0.045447,0.045447,449.7200,1.495687e+09,1.618322e+09,3325819,336110,3711,N,NACT,2025


In [5]:
crsp_wrds_msfv2_query_1990=pd.read_parquet('../data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow',columns=['permno', 'mthcaldt', 'mthret', 'mthprc', 'mthcap', 'siccd'])

# ----------------------------------------
# 剔除金融行业 (SIC 代码 6000 至 6999) 暂时作废
# crsp_wrds_msfv2_query_1990['siccd']=crsp_wrds_msfv2_query_1990['siccd'].astype('Int64')
# crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990[(crsp_wrds_msfv2_query_1990['siccd'] < 6000) | (crsp_wrds_msfv2_query_1990['siccd'] > 6999)]
# crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990.drop(columns=['siccd'])
# ----------------------------------------

print(f"total rows: {len(crsp_wrds_msfv2_query_1990)}")
crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990.drop_duplicates(subset=['permno', 'mthcaldt'], keep='first')
print(f"total rows after drop duplicates: {len(crsp_wrds_msfv2_query_1990)}")
pd.concat([crsp_wrds_msfv2_query_1990.head(3), crsp_wrds_msfv2_query_1990.tail(3)], axis=0)

total rows: 2198474
total rows after drop duplicates: 2192931


,permno,mthcaldt,mthret,mthprc,mthcap,siccd
0,10001,1990-01-31,-0.018519,9.9375,1.015613e+04,4920
1,72996,1990-01-31,-0.106383,5.2500,1.897875e+04,6211
2,72988,1990-01-31,-0.076923,1.5000,2.658000e+03,2099
2198471,21118,2025-12-31,-0.040892,2.5800,2.884208e+05,5137
2198472,84397,2025-12-31,0.036761,51.7200,5.192171e+05,6022
2198473,93436,2025-12-31,0.045447,449.7200,1.495687e+09,3711


In [6]:
MicroFactor = pd.read_excel('../data/MicroFactor.xlsx', sheet_name='select')
display(pd.concat([MicroFactor.head(3), MicroFactor.tail(3)], axis=0))

selected_features = MicroFactor[MicroFactor['select']==1]['nullable'].tolist()
comp_wrds_ratios = pd.read_parquet('../data/comp_wrds_ratios.parquet', engine='pyarrow', 
                                   columns=selected_features)
display(pd.concat([comp_wrds_ratios.head(3), comp_wrds_ratios.tail(3)], axis=0))
print("Total duplicates found: " + str(comp_wrds_ratios.duplicated(subset=['gvkey', 'public_date'], keep=False).sum()))

,name,nullable,type,comment,description,select,中文解释
0,0,gvkey,True,VARCHAR(7),Global Company Key,1.0,NaN
1,1,adate,True,DATE,Fiscal year end,NaN,NaN
2,2,qdate,True,DATE,Fiscal quarter end,NaN,NaN
92,92,ffi48,True,DOUBLE PRECISION,Fama and French 48 Industries,NaN,NaN
93,93,ffi49_desc,True,VARCHAR(5),Fama and French 49 Industries - Description,NaN,NaN
94,94,ffi49,True,DOUBLE PRECISION,Fama and French 49 Industries,1.0,NaN


,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,ffi49
0,001003,1990-01-31,NaN,-2.836651,-0.052521,0.034461,-0.159702,-0.139973,-0.276872,-0.153649,0.314643,-0.217813,-1.813523,-1.166551,0.488989,0.413638,-1.832834,0.984159,-0.128283,17.424393,0.056425,0.960262,1.554107,11.552599,11.534909,28.633479,0.000000,0.064253,0.0,-0.355052,43.0
1,013343,1990-01-31,3.597226,6.485974,4.511111,1.409560,2.241353,2.241353,0.299322,0.299322,0.299322,0.630680,0.041494,0.000000,0.040907,0.009276,0.085077,1.000000,NaN,0.024770,NaN,NaN,0.136665,47.824790,12.759404,0.140050,0.000000,0.003948,0.0,-0.044903,48.0
2,002484,1990-01-31,0.715879,NaN,11.229942,0.360000,-17.721272,NaN,0.032242,0.047622,NaN,NaN,0.116036,NaN,NaN,0.135434,NaN,0.689936,-0.090208,0.900549,0.201836,1.769802,1.870810,99.869707,NaN,3.555566,0.000000,NaN,NaN,0.099620,43.0
2610317,011600,2026-03-31,0.387288,24.216030,24.594340,2.060824,12.178577,16.074997,0.084571,0.060467,0.321403,0.124571,0.103696,0.046713,0.234632,0.194237,2.623066,0.020752,1.107575,0.808465,1.070803,3.386238,0.730025,5.656074,8.812555,1.320899,0.000000,0.024472,0.0,-0.062229,19.0
2610318,074646,2026-03-31,0.470090,13.483320,-31.896552,3.817789,9.156156,9.225138,-0.115252,0.352168,0.916135,-0.094985,-0.103688,-0.163178,0.236191,0.371024,3.864098,0.988021,0.191975,1.303163,0.569129,1.609304,0.394388,1.788370,0.153418,0.593785,0.000000,0.000000,0.0,-0.209899,NaN
2610319,027928,2026-03-31,0.137635,29.715752,28.022035,5.963023,17.182012,17.543792,0.215695,0.270861,0.817603,0.256796,0.225025,0.178667,0.466616,0.194825,1.094273,0.083494,0.767637,0.803099,0.478110,1.379504,0.570711,8.660712,4.304985,1.029049,0.158392,0.111518,0.0,-0.076517,36.0


Total duplicates found: 0


In [7]:
# merge crsp_wrds_msfv2_query_1990 and comp_wrds_ratios by ccm link
with sqlite3.connect(wrds_path) as db_conn:
    crsp_ccmxpf_linktable = pd.read_sql('SELECT * FROM crsp_ccmxpf_lnkhist', db_conn, 
                                        parse_dates=['linkdt','linkenddt'])
crsp_ccmxpf_linktable['linkenddt'] = pd.to_datetime(crsp_ccmxpf_linktable['linkenddt']).fillna(pd.to_datetime('2099-12-31'))
crsp_ccmxpf_linktable['lpermno'] = crsp_ccmxpf_linktable['lpermno'].astype("Int64").astype(str).replace("<NA>", pd.NA).str.strip()
crsp_ccmxpf_linktable['gvkey'] = crsp_ccmxpf_linktable['gvkey'].astype(str).str.zfill(6)

crsp_wrds_msfv2_query_1990['permno'] = crsp_wrds_msfv2_query_1990['permno'].astype("Int64").astype(str).replace("<NA>", pd.NA).str.strip()
comp_wrds_ratios['gvkey'] = comp_wrds_ratios['gvkey'].astype(str).str.zfill(6)

ccm = crsp_ccmxpf_linktable[(crsp_ccmxpf_linktable['linktype'].isin(['LU', 'LC'])) & (crsp_ccmxpf_linktable['linkprim'].isin(['P', 'C']))]
ccm = ccm.rename(columns={'lpermno': 'permno'})

comp_ccm = pd.merge(comp_wrds_ratios, ccm[['gvkey', 'permno', 'linkdt', 'linkenddt']], on='gvkey', how='inner')
comp_ccm = comp_ccm[(comp_ccm['public_date'] >= comp_ccm['linkdt']) & 
                    (comp_ccm['public_date'] <= comp_ccm['linkenddt'])]

comp_ccm['public_date'] = comp_ccm['public_date'] + pd.offsets.MonthEnd(0) # 把日期推到月底，和CRSP日期对齐
crsp_wrds_msfv2_query_1990['mthcaldt'] = crsp_wrds_msfv2_query_1990['mthcaldt'] + pd.offsets.MonthEnd(0)# 把日期推到月底
print(f"CRSP monthly data shape: {crsp_wrds_msfv2_query_1990.shape}")
# 匹配每一行有交易数据的股票和财报数据，注意这里是 left join，保留所有的 CRSP 行，财报数据不匹配的会被填 NaN
micro_df = pd.merge(
    crsp_wrds_msfv2_query_1990,
    comp_ccm,
    left_on=['permno', 'mthcaldt'],
    right_on=['permno', 'public_date'],
    how='left'
)
micro_df = micro_df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)

# =========================================================
# 核心修复：强制对齐“日历月网格” (Resampling to Continuous Grid)
# =========================================================
print(" -> Reindexing panel data to a continuous monthly grid...")

# 2. 将日期设置为 Index，为 Pandas 的重采样做准备
micro_df = micro_df.set_index('mthcaldt')

# 3. ★ 核心魔法：按 permno 分组，按月 ('ME' 或 'M') 重新采样
# 只要两个日期之间有断层，Pandas 就会自动插入缺失的月份，并将那一行的值设为 NaN
micro_df = micro_df.groupby('permno').resample('ME').asfreq().drop(columns=['permno']).reset_index()
micro_df = micro_df.drop(columns=['linkdt', 'linkenddt', 'public_date'], errors='ignore') # 删除不再需要的列
pd.concat([micro_df.head(5), micro_df.tail(5)], axis=0)

CRSP monthly data shape: (2192931, 6)
 -> Reindexing panel data to a continuous monthly grid...


,permno,mthcaldt,mthret,mthprc,mthcap,siccd,gvkey,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,ffi49
0,10001,1990-01-31,-0.018519,9.9375,1.015613e+04,4920.0,012994,0.917331,4.853678,2.743056,0.421441,7.769246,7.599586,0.050236,0.103437,0.134464,0.081263,0.151031,0.139773,0.164538,0.426239,2.590528,0.107954,0.339098,2.235379,0.271814,1.173138,1.223659,7.855339,10.165835,3.959000,0.000000,0.0,0.0,-0.005309,31.0
1,10001,1990-02-28,-0.006289,9.8750,1.009225e+04,4920.0,012994,0.849363,4.626443,2.544529,0.403697,5.646409,7.695783,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,2.084969,0.273871,1.243238,1.301493,8.324893,10.832774,4.015067,0.000000,0.0,0.0,-0.024148,31.0
2,10001,1990-03-31,0.012821,9.8750,1.014163e+04,4920.0,012994,0.849363,4.626443,2.576336,0.410743,5.744959,7.830102,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,2.084969,0.273871,1.243238,1.301493,8.324893,10.832774,4.015067,0.000000,0.0,0.0,-0.024148,31.0
3,10001,1990-04-30,0.000000,9.8750,1.014163e+04,4920.0,012994,0.849363,4.626443,2.544529,0.405672,5.674033,7.733434,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,2.084969,0.273871,1.243238,1.301493,8.324893,10.832774,4.015067,0.000000,0.0,0.0,-0.024148,31.0
4,10001,1990-05-31,-0.012658,9.7500,1.001325e+04,4920.0,012994,0.911008,4.819588,2.480916,0.422964,4.424768,7.540098,0.056644,0.116415,0.148982,0.089212,0.158487,0.139023,0.182316,0.403169,2.211369,0.066927,0.693056,1.943177,0.263992,1.278309,1.223747,8.168377,10.663666,3.601704,0.000000,0.0,0.0,-0.047016,31.0
2226539,93436,2025-08-31,0.083044,333.8700,1.076881e+09,3711.0,184996,0.075492,88.014082,199.922156,11.609112,68.277633,192.695467,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.670667,1.224750,1.974345,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,23.0
2226540,93436,2025-09-30,0.332015,444.7200,1.478249e+09,3711.0,184996,0.075492,88.014082,266.299401,15.943154,93.767794,264.634672,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.670667,1.224750,1.974345,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,23.0
2226541,93436,2025-10-31,0.026624,456.5600,1.518436e+09,3711.0,184996,0.075492,88.014082,273.389222,16.367617,96.264221,271.680172,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.670667,1.224750,1.974345,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,23.0
2226542,93436,2025-11-30,-0.057802,430.1700,1.430668e+09,3711.0,184996,0.054098,107.865207,298.729167,14.951796,90.797884,209.231062,0.053141,0.050903,0.232817,0.118495,0.068929,0.064976,0.174805,0.105348,1.234543,0.225793,0.525494,0.661299,1.285071,2.031625,0.750824,22.058125,5.472834,1.254820,0.061736,0.0,0.0,-0.084740,23.0
2226543,93436,2025-12-31,0.045447,449.7200,1.495687e+09,3711.0,184996,0.054098,107.865207,312.305556,17.639306,107.118346,246.839292,0.053141,0.050903,0.232817,0.118495,0.068929,0.064976,0.174805,0.105348,1.234543,0.225793,0.525494,0.661299,1.285071,2.031625,0.750824,22.058125,5.472834,1.254820,0.061736,0.0,0.0,-0.084740,23.0


In [8]:
# 按月统计全部股票每一列缺失的占比, 只打印超过20%的月份和列
missing_by_month = micro_df.groupby('mthcaldt').apply(lambda x: x.isna().mean())
missing_by_month = missing_by_month[missing_by_month > 0.2].stack().reset_index() #stack() 将多层索引的 DataFrame 转换为长格式
missing_by_month.columns = ['mthcaldt', 'column', 'missing_ratio']
# print("Months and columns with more than 20% missing values:")
# print(missing_by_month)
missing_by_month.to_excel('../data/missing_by_month.xlsx', index=False)
# 主要是divyield缺失比较严重, 已经删除了缺失值很严重的数据

/var/folders/8b/0krd96r57jldyg4t2s79cv140000gn/T/ipykernel_2813/2301910022.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_by_month = micro_df.groupby('mthcaldt').apply(lambda x: x.isna().mean())


In [9]:
# =========================================================
# 只在 micro_df 上完成需要原始微观变量的特征工程
# 注意：必须先完成 target / momentum / rolling features，之后才能丢弃原始微观列。
# =========================================================

micro_exclude_cols = {
    'permno', 'mthcaldt', 'mthprc', 'siccd', 'gvkey', 'ffi49'
}

micro_cols = [
    col for col in micro_df.columns
    if col not in micro_exclude_cols
]

micro_df = micro_df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
# 1. 计算 t+1 收益率
micro_df['target_ret_final'] = micro_df.groupby('permno')['mthret'].shift(-1)

# 2. 计算过去 12 个月动量（跳过近 1 月，滚动 11 月）
micro_df['log_ret'] = np.log1p(micro_df['mthret'])
micro_df['MOM12m'] = micro_df.groupby('permno')['log_ret'].transform(
    lambda x: x.shift(1).rolling(window=11, min_periods=8).sum()
)
micro_df['MOM12m'] = np.exp(micro_df['MOM12m']) - 1
micro_df = micro_df.drop(columns=['log_ret'])

# 3. 在完整微观时间序列上计算 12 个月滚动均值和标准差
micro_df['mu_bench'] = micro_df.groupby('permno')['mthret'].transform(
    lambda x: x.rolling(12, min_periods=8).mean()
)
micro_df['sigma_bench'] = micro_df.groupby('permno')['mthret'].transform(
    lambda x: x.rolling(12, min_periods=8).std()
)

derived_micro_cols = ['MOM12m', 'mu_bench', 'sigma_bench']
micro_cols = micro_cols + derived_micro_cols

# 剔除价格低于 5 美元的股票
micro_df = micro_df[micro_df['mthprc'] >= 5].copy()

# 先用“月份 + 行业”中位数填充，再用当月全市场中位数补充, 最后用0填充缺失值
for col in micro_cols:
    industry_median = micro_df.groupby(['mthcaldt', 'ffi49'])[col].transform('median')
    market_median = micro_df.groupby('mthcaldt')[col].transform('median')
    micro_df[col] = micro_df[col].fillna(industry_median).fillna(market_median).fillna(0)

# 对mthcap和mthvol取log
micro_df['mthcap_log'] = np.log(micro_df['mthcap'])

# 将 ffi49 缺失值填充为 0，并转换为整数类型
micro_df['ffi49'] = micro_df['ffi49'].fillna(0).astype(int)


micro_df = micro_df[
    micro_df['mthcaldt'].between('1991-01-31', '2024-12-31')
].reset_index(drop=True)

# 删除缺少 target_ret_final 的记录
micro_df = micro_df.dropna(
    subset=['target_ret_final']
).reset_index(drop=True)

pd.concat([micro_df.head(5), micro_df.tail(5)])

/Users/jianbinchen/miniconda3/envs/research/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,permno,mthcaldt,mthret,mthprc,mthcap,siccd,gvkey,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,ffi49,target_ret_final,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1991-01-31,0.013158,9.625,1.014475e+04,4920.0,012994,0.896385,5.690808,3.307560,0.445982,4.872598,14.070388,0.044401,0.099749,0.138304,0.082956,0.112235,0.112708,0.161306,0.392076,2.430626,0.061202,0.672101,1.796365,0.316439,1.340247,1.166318,8.789413,10.931698,3.261452,0.000000,0.0,0.0,-0.055775,31,0.012987,0.009425,0.002110,0.022682,9.224712
1,10001,1991-02-28,0.012987,9.750,1.027650e+04,4920.0,012994,0.975332,5.299653,3.037383,0.434947,5.205927,14.253121,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,-0.012244,0.029179,0.003717,0.022716,9.237615
2,10001,1991-03-31,-0.012244,9.500,1.001300e+04,4920.0,012994,0.975332,5.299653,2.959502,0.423795,5.072442,13.887656,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.039474,0.029348,0.001628,0.022954,9.211640
3,10001,1991-04-30,0.039474,9.875,1.040825e+04,4920.0,012994,0.975332,5.299653,3.115265,0.446100,5.339412,14.618585,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.000000,0.016744,0.004917,0.025398,9.250354
4,10001,1991-05-31,0.000000,9.875,1.040825e+04,4920.0,012994,1.040348,5.316239,3.086420,0.444182,6.893394,14.618585,0.047832,0.103923,0.139492,0.083400,0.119963,0.115336,0.165372,0.375721,2.271979,0.059539,0.472059,1.708091,0.318617,1.389858,1.185531,8.331080,10.836074,3.210526,0.000000,0.0,0.0,-0.019640,31,0.079241,0.070429,0.005972,0.024859,9.250354
1560355,93436,2024-08-31,-0.077390,214.110,6.840044e+08,3711.0,184996,0.105166,57.713252,59.310249,7.174588,59.301712,335.706250,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.221942,-0.100783,-0.006184,0.139169,20.343475
1560356,93436,2024-09-30,0.221942,261.630,8.390474e+08,3711.0,184996,0.105166,57.713252,72.473684,8.802612,72.758187,346.216280,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,-0.045025,-0.144312,0.014849,0.153502,20.547778
1560357,93436,2024-10-31,-0.045025,249.850,8.020335e+08,3711.0,184996,0.105166,57.713252,69.210526,8.406271,69.482219,369.690039,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.381469,0.302679,0.027542,0.140071,20.502661
1560358,93436,2024-11-30,0.381469,345.160,1.107984e+09,3711.0,184996,0.083346,53.762150,93.286486,11.394010,76.450592,306.628288,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.170008,0.040696,0.043050,0.167882,20.825808
1560359,93436,2024-12-31,0.170008,403.840,1.298958e+09,3711.0,184996,0.083346,53.762150,109.145946,13.368497,89.698836,274.076290,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.001882,0.389088,0.054301,0.171772,20.984828


In [10]:
# micro_df.drop(columns=[ 'public_date', 'linkdt', 'linkenddt', 'log_ret'], errors='ignore', inplace=True)
micro_df.to_parquet('../data/micro_data.parquet', index=False)

In [ ]:
import matplotlib.pyplot as plt
# draw the distribution of mthret in micro_df, range from -1 to 1, with 500 bins
plt.figure(figsize=(10, 6))
plt.hist(micro_df['mthret'], bins=500, range=(-0.5, 0.5), color='skyblue', edgecolor='black')
plt.title('Distribution of mthret')
plt.xlabel('mthret')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# draw the distribution of mthret in processed_df, range from -1 to 1, with 500 bins
plt.figure(figsize=(10, 6))
plt.hist(processed_df['target_ret_final'], bins=500, range=(-0.5, 0.5), color='skyblue', edgecolor='black')
plt.title('Distribution of target_ret_final')
plt.xlabel('target_ret_final')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 1. 确保日期字段是 datetime 类型
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt'])
processed_df['mthcaldt'] = pd.to_datetime(processed_df['mthcaldt'])

# 2. 统计每月样本数量
counts_raw = micro_df['mthcaldt'].value_counts().sort_index()
counts_proc = processed_df['mthcaldt'].value_counts().sort_index()

# ★ 核心修正：利用 DataFrame 的对齐机制合并两个序列，防止时间轴错位
plot_df = pd.DataFrame({
    'raw': counts_raw,
    'proc': counts_proc
}).fillna(0)  # 如果某个月全被删光了，清洗后的数量记为 0

# 3. 开始构图
plt.figure(figsize=(13, 6))

# 画原始数据的波动线（使用对齐后的索引和值）
plt.plot(plot_df.index, plot_df['raw'], 
         label='Raw Data (micro_df)', color='#7f8c8d', linestyle='--', linewidth=1.5)

# 画清洗后数据的波动线
plt.plot(plot_df.index, plot_df['proc'], 
         label='Processed Data (processed_df)', color='#2c3e50', linewidth=2.5)

# 4. 核心亮点：此时两个 Series 共享完全一致的 Index，阴影绝对不会错位
plt.fill_between(plot_df.index, plot_df['proc'], plot_df['raw'], 
                 color='#e74c3c', alpha=0.3, label='Dropped Observations (NaN / Outliers)')

# 5. 学术规范的图表点缀
plt.title('Sample Coverage Comparison Over Time (1990–2024)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Investment Date (mthcaldt)', fontsize=12)
plt.ylabel('Number of Active Stocks (Cross-sectional Count)', fontsize=12)

plt.legend(loc='best', fontsize=11, frameon=True, shadow=False)
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:

sector_order = {
    'Tech & Telecom': 1,
    'Financials': 2,
    'Consumer & Health': 3,
    'Cyclical & Heavy': 4,
    'Services & Other': 5
}
# =========================================================
# 2. 数字编码 → 描述字符串映射
#    ffi49_pairs 结构：ffi49（数字）| ffi49_desc（字符串如 'RTAIL'）
# =========================================================
ffi49_desc_mapping = dict(zip(ffi49_pairs['ffi49'], ffi49_pairs['ffi49_desc']))

# =========================================================
# 3. 统计各行业的股票数量（用 -999 占位缺失值）
# =========================================================
industry_counts_raw  = micro_df['ffi49'].fillna(0).value_counts()
industry_counts_proc = processed_df['ffi49'].fillna(0).value_counts()

all_industry_codes = sorted(
    set(industry_counts_raw.index).union(set(industry_counts_proc.index))
)

# =========================================================
# 4. 构建排序 DataFrame
#    关键逻辑：数字码 → 描述字符串 → 大写 → 查板块字典
# =========================================================
sort_data = []

for code in all_industry_codes:
    # Step 1：数字码 → 描述（缺失值单独处理）
    # if code == -999:
    #     desc = 'NoTag'
    # else:
    #     desc = ffi49_desc_mapping.get(code, 'NoTag')  # 找不到也兜底为 NoTag
    desc = ffi49_desc_mapping[code]
    # Step 2：描述字符串大写 → 查板块字典
    sector = macro_sector_mapping[str(desc).upper()]

    sort_data.append({
        'Industry_Code': code,
        'Industry_Desc': desc,
        'Sector':        sector,
        'Sector_Rank':   sector_order[sector],
        'Raw_Count':     industry_counts_raw[code],
        'Proc_Count':    industry_counts_proc[code],
    })

df_sort = pd.DataFrame(sort_data)

# =========================================================
# 5. 排序：先按板块（Sector_Rank 升序），再按数量（Raw_Count 降序）
# =========================================================
df_sort = (
    df_sort
    .sort_values(by=['Sector_Rank', 'Raw_Count'], ascending=[True, False])
    .reset_index(drop=True)
)

# =========================================================
# 6. 绘图
# =========================================================
fig, ax = plt.subplots(figsize=(20, 8))

x     = np.arange(len(df_sort))
width = 0.72

# 底层：原始数据（浅红，透明）
ax.bar(x, df_sort['Raw_Count'],  width=width, label='Raw Data (Before Cleaning)',  color='#e74c3c', alpha=0.35)
# 顶层：保留数据（深蓝，不透明）
ax.bar(x, df_sort['Proc_Count'], width=width, label='Processed Data (Retained)', color='#2c3e50', alpha=0.90)

ax.set_title('Industry Coverage Before and After Data Cleaning', fontsize=18, fontweight='bold', pad=28)
ax.set_xlabel('Industry (FFI49)', fontsize=13)
ax.set_ylabel('Number of Stocks', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(df_sort['Industry_Desc'].tolist(), rotation=45, ha='right', fontsize=10)

# =========================================================
# 7. 板块分隔线 + 板块名称标注
#    drop_duplicates 按 Sector_Rank 去重，取每个板块第一行的行号
# =========================================================
sector_boundaries = (
    df_sort
    .drop_duplicates(subset=['Sector_Rank'])
    .index
    .tolist()
)

max_y = df_sort['Raw_Count'].max()

for idx, start_idx in enumerate(sector_boundaries):

    # 分隔虚线（第一个板块左侧不画线）
    if idx > 0:
        ax.axvline(x=start_idx - 0.5, color='gray', linestyle='--', alpha=0.6)

    # 当前板块结束位置
    end_idx = sector_boundaries[idx + 1] if idx + 1 < len(sector_boundaries) else len(df_sort)
    center_x   = (start_idx + end_idx - 1) / 2
    sector_name = df_sort.loc[start_idx, 'Sector']

    ax.text(
        center_x, max_y * 1.03, sector_name,
        ha='center', va='bottom', fontsize=11, fontweight='bold', color='#34495e',
        bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', pad=2)
    )

ax.set_ylim(0, max_y * 1.15)
ax.grid(axis='y', linestyle=':', alpha=0.5)
ax.legend(loc='upper right', fontsize=11, frameon=True)

plt.tight_layout()
plt.savefig('../paper/fig/Industry_Coverage_Comparison.pdf', format='pdf', bbox_inches='tight')
plt.show()